In [1]:
# Install required libraries
!pip install duckdb pandas scikit-learn lightgbm matplotlib -q

print("Libraries installed successfully!")

Libraries installed successfully!


In [3]:
import duckdb
import pandas as pd
import numpy as np

# --- Set your Hugging Face Token here ---
HF_TOKEN = 'HF_TOKEN = 'YOUR_HF_TOKEN_HERE''

# Initialize DuckDB connection and load remote reading tools
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")

# Set Hugging Face secret using updated DuckDB syntax
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

print("DuckDB connection established successfully!")

DuckDB connection established successfully!


In [5]:
# Querying performance data from the FlyRank warehouse to extract key features
query = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) as avg_impressions,
    AVG(gsc_clicks) as avg_clicks,
    AVG(gsc_avg_position) as avg_position,
    STDDEV(gsc_clicks) as click_volatility,
    COUNT(report_date) as active_days,
    -- Labeling decay: 1 if average daily clicks drop under threshold, else 0
    CASE WHEN AVG(gsc_clicks) < 5 THEN 1 ELSE 0 END as needs_refresh
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
GROUP BY client_hash_id, content_hash_id
HAVING COUNT(report_date) > 5
LIMIT 50000;
"""

print("Processing dataset sample and building features (takes ~20 seconds)...")
df = con.execute(query).df()
print(f"Data processed successfully! Total rows loaded: {len(df)}")

Processing dataset sample and building features (takes ~20 seconds)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data processed successfully! Total rows loaded: 50000


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report

# 1. Define a clear target: High-ranking positions (< 20) vs low-ranking positions (>= 20)
# 1 = High visibility / Healthy, 0 = Low visibility / Needs Refresh
df['needs_refresh'] = (df['avg_position'] >= 20.0).astype(int)

# 2. Select independent features (excluding position/clicks from inputs to prevent leakage)
feature_cols = ['avg_impressions', 'click_volatility', 'active_days']
X = df[feature_cols].fillna(0)
y = df['needs_refresh']

# 3. Perform an 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Fit the Gradient Boosting Classifier
model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate predictions
y_preds = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_preds)

print("--- MODEL TRAINING COMPLETE ---")
print(f"Model AUC-ROC Score: {auc_score:.4f} (Baseline = 0.5000)")
print("\nDetailed Performance Report:")
print(classification_report(y_test, (y_preds > 0.5).astype(int)))

--- MODEL TRAINING COMPLETE ---
Model AUC-ROC Score: 0.9283 (Baseline = 0.5000)

Detailed Performance Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      9255
           1       0.45      0.02      0.05       745

    accuracy                           0.93     10000
   macro avg       0.69      0.51      0.50     10000
weighted avg       0.89      0.93      0.89     10000



In [9]:
# Export summary metrics into a local CSV file for artifact verification
summary_data = df[['client_hash_id', 'content_hash_id', 'avg_impressions', 'avg_position', 'needs_refresh']].head(1000)
summary_data.to_csv("capstone_summary.csv", index=False)

print("Export complete! Saved capstone_summary.csv locally.")

Export complete! Saved capstone_summary.csv locally.
